# Analyzing preprocessing results

This notebook loads the snirf files containing the blockaverages and plots the estimated HRF for each trial_type in two channels on the left and right hemisphere.

In [ ]:
import cedalion.io
import matplotlib.pyplot as p
from pathlib import Path

#print(snakemake.input)

fnames = list(snakemake.input)

f, ax = p.subplots(len(fnames), 2, figsize=(16, 3*len(fnames)))
channels = ["S6D12", "S13D18"]

for i_file, fname in enumerate(fnames):
    rec = cedalion.io.read_snirf(fname)[0]

    # the block averaged time series is stored under a name prefixed with 'hrf'
    ts_name = next(k for k in rec.timeseries if k.startswith("hrf"))
    hrf = rec[ts_name]


    for i_ch, channel in enumerate(channels):
        #channel = hrf.channel.values[0]
        tmp = hrf.sel(channel=channel)

        for trial_type in tmp.trial_type.values:
            tt = tmp.sel(trial_type=trial_type)
            ax[i_file, i_ch].plot(tt.time, tt.sel(chromo="HbO"), "r-", label=f"HbO {trial_type}")
            ax[i_file, i_ch].plot(tt.time, tt.sel(chromo="HbR"), "b-", label=f"HbR {trial_type}")

        ax[i_file, i_ch].axvline(0., c="k", ls=":")
        ax[i_file, i_ch].set_title(f"{Path(fname).name} - Ch. {channel}")
        ax[i_file, i_ch].set_xlabel("time / s")
        ax[i_file, i_ch].legend(ncol=2, fontsize="small")
        ax[i_file, i_ch].set_ylim(-15,40)

p.tight_layout()
f.savefig(snakemake.output.output_overview)